# scrollbar

> Custom scrollbar component with proportional thumb for position indication.

In [ ]:
#| default_exp components.scrollbar

In [ ]:
#| export
from fasthtml.common import Div

from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.layout import position, display_tw
from cjm_fasthtml_tailwind.utilities.interactivity import cursor, select
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

from cjm_fasthtml_virtual_scrollbar.core.models import ScrollbarConfig, ScrollbarState, ScrollbarIds
from cjm_fasthtml_virtual_scrollbar.core.math import compute_scrollbar

## render_scrollbar_thumb

In [ ]:
#| export
def render_scrollbar_thumb(
    state: ScrollbarState,       # Scrollbar state
    config: ScrollbarConfig,     # Scrollbar config
    ids: ScrollbarIds,           # HTML IDs
    track_height: float = 600.0, # Track height for min thumb calculation
    oob: bool = False,           # Whether to include hx-swap-oob
) -> Div:  # Thumb element
    """Render the scrollbar thumb at the correct position."""
    thumb_top, thumb_height = compute_scrollbar(
        state.position, state.visible_count, state.total_items,
        track_height, config.min_thumb_height,
    )

    thumb = Div(
        id=ids.thumb,
        cls=combine_classes(
            position.absolute, w.full, border_radius.field,
            bg_dui.base_content.opacity(30),
            cursor.grab,
        ),
        style=f"top: {thumb_top:.2f}%; height: {thumb_height:.2f}%",
    )

    if oob:
        thumb.attrs["hx-swap-oob"] = "outerHTML"

    return thumb

## render_scrollbar

In [ ]:
#| export
def _build_track_cls(track_width: int) -> str:  # Combined CSS class string
    """Build track CSS classes for a given width."""
    return combine_classes(
        w(track_width), position.relative, border_radius.field,
        bg_dui.base_content.opacity(10),
        cursor.pointer, select.none,
    )

In [ ]:
#| export
def render_scrollbar(
    state: ScrollbarState,       # Scrollbar state
    config: ScrollbarConfig,     # Scrollbar config
    ids: ScrollbarIds,           # HTML IDs
) -> Div:  # Complete scrollbar (track + thumb)
    """Render the custom scrollbar with track and proportional thumb."""
    # Hide track when all items visible, but keep element for consistent layout
    if state.total_items <= state.visible_count:
        return Div(id=ids.track, cls=str(display_tw.hidden))

    thumb = render_scrollbar_thumb(state, config, ids)

    return Div(
        thumb,
        id=ids.track,
        cls=_build_track_cls(config.track_width),
        data_total_items=str(state.total_items),
        data_visible_count=str(state.visible_count),
    )

## Tests

In [ ]:
from fasthtml.common import to_xml

In [ ]:
# Hidden when all items visible
state = ScrollbarState(position=0, visible_count=10, total_items=5)
config = ScrollbarConfig(prefix="test")
ids = ScrollbarIds(prefix="test")
html = to_xml(render_scrollbar(state, config, ids))
assert 'hidden' in html
assert 'test-scrollbar-track' in html

In [ ]:
# Visible with thumb when items exceed visible
state = ScrollbarState(position=0, visible_count=10, total_items=100)
html = to_xml(render_scrollbar(state, config, ids))
assert 'test-scrollbar-track' in html
assert 'test-scrollbar-thumb' in html
assert 'data-total-items="100"' in html
assert 'data-visible-count="10"' in html
assert 'hidden' not in html

In [ ]:
# OOB thumb
thumb = render_scrollbar_thumb(state, config, ids, oob=True)
html = to_xml(thumb)
assert 'hx-swap-oob="outerHTML"' in html